# Linework and cross-sections

This is the geometry under breaklines, alignments, and cross-sections.

Station here is **horizontal** chainage in XY, not 3D slope distance. Offset is **positive to the right** of increasing station: a road going east has south on its right. For a tangent `(tx, ty)`, that right-hand normal is `(ty, -tx)`.

Confirm both conventions if someone defines them differently. Do not use Z when computing station.


These cells are PyTorch. Geometry wants float64, so the setup cell calls `torch.set_default_dtype(torch.float64)`. The library default is float32, and a plane SVD or a Kabsch in float32 will miss a hand check that passes in float64.


In [ ]:
import torch

torch.set_default_dtype(torch.float64)
torch.set_printoptions(precision=4, sci_mode=False)


## 1. Station and signed offset

Project XY points onto a polyline of XY vertices. Return station, signed offset, and the closest point on the polyline.

Station is the distance along the polyline to that closest point, so it lies in `[0, length]`. Offset is positive to the right of the segment that won. Ties go to the smaller segment index, and the sign then follows that segment's tangent.

Past either end, the closest point is the endpoint, so station is clamped. The offset you return is the lateral component of `(point - endpoint)`, which is not the Euclidean distance. Say that when you talk through the function. A corner can belong to two segments; say the tie-break too.

Ignore Z. These inputs are `(N, 2)` and `(M, 2)`.

Hand check: polyline `(0,0) → (10,0) → (10,10)`.

| point | station | offset | why |
| --- | --- | --- | --- |
| `(5, 2)` | 5 | −2 | two meters north, which is left of an eastbound segment |
| `(12, 5)` | 15 | +2 | two meters east of the northbound segment |
| `(8, 5)` | 15 | −2 | two meters west of that segment |
| `(10, 10)` | 20 | 0 | the end vertex |
| `(5, 0)` | 5 | 0 | on the line |

<details><summary>Hint</summary>

Broadcast points against segments: `t = clip((P - A) · AB / ||AB||², 0, 1)`, closest = `A + t AB`, pick the minimum distance. Right normal of a unit tangent is `(ty, -tx)`. Offset is the dot product of `(point - closest)` with that normal. Zero-length segments need a guard.

</details>


In [ ]:
import torch

def station_offset(points_xy, polyline_xy):
    """
    Horizontal station and signed offset of points against a polyline.

    Station is chainage along the polyline in the XY plane, in [0, length],
    measured to the closest point on the finite polyline.
    Offset is positive to the right of the winning segment's direction of
    travel. Tangent (tx, ty) has right-hand normal (ty, -tx): a road going
    east has south on its right.

    Past either end, station is clamped, so offset is only the lateral part of
    the vector to the endpoint, not the Euclidean distance. Corners can tie;
    the lowest segment index wins, and the sign then follows that segment.

    points_xy: (N, 2), polyline_xy: (M, 2), M >= 2.
    returns station (N,), offset (N,), closest (N, 2).
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def station_offset(points_xy, polyline_xy):
    """Horizontal station and signed offset against a polyline.

    Offset is positive to the right. Tangent (tx, ty) has right normal (ty, -tx).
    points_xy: (N, 2), polyline_xy: (M, 2).
    returns station (N,), offset (N,), closest (N, 2).
    """
    points_xy = _t(points_xy)
    polyline_xy = _t(polyline_xy)
    if polyline_xy.shape[0] < 2:
        raise ValueError("polyline needs at least 2 vertices")
    A = polyline_xy[:-1]
    B = polyline_xy[1:]
    AB = B - A
    L2 = (AB * AB).sum(dim=1)
    L = torch.sqrt(L2)
    AP = points_xy[:, None, :] - A[None, :, :]
    denom = torch.clamp(L2, min=1e-30)
    t = (AP * AB[None, :, :]).sum(dim=-1) / denom
    t = torch.clamp(t, 0.0, 1.0)
    closest = A[None, :, :] + t[..., None] * AB[None, :, :]
    zero = L < 1e-12
    if bool(zero.any()):
        closest[:, zero, :] = A[zero]
        t[:, zero] = 0.0
    delta = points_xy[:, None, :] - closest
    dist2 = (delta * delta).sum(dim=-1)
    seg = dist2.argmin(dim=1)
    rows = torch.arange(points_xy.shape[0])
    t_best = t[rows, seg]
    closest_best = closest[rows, seg]
    cum = torch.cat((torch.zeros(1, dtype=points_xy.dtype), torch.cumsum(L, dim=0)))
    station = cum[seg] + t_best * L[seg]
    tangent = torch.zeros_like(AB)
    good = ~zero
    tangent[good] = AB[good] / L[good, None]
    right = torch.stack((tangent[:, 1], -tangent[:, 0]), dim=1)
    offset = ((points_xy - closest_best) * right[seg]).sum(dim=1)
    return station, offset, closest_best

def check_station_offset():
    poly = torch.tensor([[0.0, 0.0], [10.0, 0.0], [10.0, 10.0]])
    pts = torch.tensor(
        [
            [5.0, 2.0],
            [12.0, 5.0],
            [8.0, 5.0],
            [10.0, 10.0],
            [5.0, 0.0],
        ]
    )
    station, offset, closest = station_offset(pts, poly)
    assert torch.allclose(station, torch.tensor([5.0, 15.0, 15.0, 20.0, 5.0]))
    assert torch.allclose(offset, torch.tensor([-2.0, 2.0, -2.0, 0.0, 0.0]))
    assert torch.allclose(closest[0], torch.tensor([5.0, 0.0]))
    assert torch.allclose(closest[1], torch.tensor([10.0, 5.0]))
    print("passed")

check_station_offset()


## 2. Cross-section

Cut a vertical section through an `(N, 3)` cloud at a horizontal station along an `(M, 3)` polyline.

The section plane is vertical and perpendicular to the horizontal tangent. Offset is horizontal and positive to the right. Elevation is absolute Z, not height above the alignment. Keep points with `|along| <= half_thickness` and `|offset| <= half_width`, and return them sorted by offset. Also return the 3D origin of the section, including Z linearly interpolated along the segment.

At an interior vertex, use the **outgoing** segment's tangent. At the last vertex, use the incoming segment. You may assume the station is on the polyline and that every segment has positive horizontal length.

Hand check: vertices `(0,0,10)`, `(10,0,20)`, `(10,10,20)`. At station 5 the origin is `(5, 0, 15)`.

- `(5, 2, 100)` has offset −2, elevation 100
- `(5.2, -1, 101)` has offset +1, elevation 101
- `(8, 0, 50)` is too far along the road if `half_thickness` is 0.5
- `(5, 30, 40)` is outside `half_width` 5

At station 10, the kink, the outgoing tangent points north. The point `(12, 0, 5)` is then on the section, offset +2. A point at `(18, 0, 5)` is outside `half_width` 5.

<details><summary>Hint</summary>

Horizontal segment lengths, cumulative station, `searchsorted` to find the segment, then origin = `A + alpha (B - A)` in 3D. `along = (xy - origin_xy) · tangent`, `offset = (xy - origin_xy) · (ty, -tx)`.

</details>


In [ ]:
import torch

def cross_section(points, polyline, station, half_thickness, half_width):
    """
    Vertical cross-section of a point cloud at a horizontal station.

    Station is XY chainage along `polyline` (M, 3). The section plane is
    vertical and perpendicular to the horizontal tangent. At an interior
    vertex the outgoing segment supplies the tangent. At the final vertex the
    incoming segment does.

    Offset is horizontal and positive to the right. Elevation is absolute Z,
    not height relative to the alignment. Points with |along| <= half_thickness
    and |offset| <= half_width are kept and sorted by offset.

    returns offsets (K,), elevations (K,), origin (3,).
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def cross_section(points, polyline, station, half_thickness, half_width):
    """Vertical cross-section at a horizontal station.

    returns offsets, elevations, origin. Sorted by offset.
    """
    points = _t(points)
    polyline = _t(polyline)
    seg = polyline[1:] - polyline[:-1]
    L = torch.linalg.norm(seg[:, :2], dim=1)
    if bool((L < 1e-12).any()):
        raise ValueError("polyline segment has zero horizontal length")
    cum = torch.cat((torch.zeros(1, dtype=points.dtype), torch.cumsum(L, dim=0)))
    total = float(cum[-1])
    s = float(station)
    if s < -1e-6 or s > total + 1e-6:
        raise ValueError("station is outside the polyline")
    s = min(max(s, 0.0), total)
    i = int(torch.searchsorted(cum, torch.tensor(s), right=True) - 1)
    i = min(max(i, 0), L.shape[0] - 1)
    if s >= total - 1e-9:
        i = L.shape[0] - 1
    alpha = float(min(max((s - float(cum[i])) / float(L[i]), 0.0), 1.0))
    origin = polyline[i] + alpha * seg[i]
    tangent = seg[i, :2] / L[i]
    right = torch.stack((tangent[1], -tangent[0]))
    delta = points[:, :2] - origin[:2]
    along = delta @ tangent
    offset = delta @ right
    mask = (along.abs() <= half_thickness) & (offset.abs() <= half_width)
    offset = offset[mask]
    elevation = points[mask, 2]
    order = torch.argsort(offset)
    return offset[order], elevation[order], origin

def check_cross_section():
    poly = torch.tensor(
        [
            [0.0, 0.0, 10.0],
            [10.0, 0.0, 20.0],
            [10.0, 10.0, 20.0],
        ]
    )
    pts = torch.tensor(
        [
            [5.0, 2.0, 100.0],
            [5.2, -1.0, 101.0],
            [8.0, 0.0, 50.0],
            [5.0, 30.0, 40.0],
        ]
    )
    offset, elev, origin = cross_section(pts, poly, 5.0, half_thickness=0.5, half_width=5.0)
    assert torch.allclose(origin, torch.tensor([5.0, 0.0, 15.0]))
    assert torch.allclose(offset, torch.tensor([-2.0, 1.0]))
    assert torch.allclose(elev, torch.tensor([100.0, 101.0]))
    offset, elev, origin = cross_section(
        torch.tensor([[12.0, 0.0, 5.0], [18.0, 0.0, 5.0]]),
        poly,
        10.0,
        half_thickness=0.5,
        half_width=5.0,
    )
    assert torch.allclose(origin, torch.tensor([10.0, 0.0, 20.0]))
    assert torch.allclose(offset, torch.tensor([2.0]))
    assert torch.allclose(elev, torch.tensor([5.0]))
    print("passed")

check_cross_section()


## 3. Point in polygon

Even-odd test. `points` is `(N, 2)`, `polygon` is `(M, 2)` without a required repeated closing vertex. Return a boolean mask.

The boundary is not a guaranteed case; test clearly inside and clearly outside points. A horizontal edge must not be counted. A vertex should be counted once, or a ray that passes through a vertex will flicker.

Hand check: square `(0,0), (4,0), (4,4), (0,4)`. `(2,2)` is inside, `(5,1)` is outside, `(-1, 0)` is outside.

A second polygon, the U `(0,0), (5,0), (5,5), (3,5), (3,2), (2,2), (2,5), (0,5)`:

- `(1, 4)` inside the left tower
- `(2.5, 4)` in the notch, outside
- `(2.5, 1)` in the base, inside
- `(4, 4)` inside the right tower

<details><summary>Hint</summary>

Cast a ray in +X. An edge straddles the point's y when exactly one endpoint is strictly above it: `(y0 > y) != (y1 > y)`. Count the edge when the intersection's x is greater than the point's x. Skip edges with `dy == 0` before dividing.

</details>


In [ ]:
import torch

def points_in_polygon(points, polygon):
    """
    Even-odd point-in-polygon test. Boundary points are not guaranteed.

    points: (N, 2), polygon: (M, 2). Repeating the first vertex at the end is
    optional. A horizontal ray in +X counts an edge when one endpoint is
    strictly above the point and the other is not, so a vertex is counted once.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def points_in_polygon(points, polygon):
    """Even-odd point-in-polygon test. Boundary points are not guaranteed."""
    points = _t(points)
    polygon = _t(polygon)
    x = points[:, 0]
    y = points[:, 1]
    x0 = polygon[:, 0]
    y0 = polygon[:, 1]
    x1 = torch.roll(x0, -1)
    y1 = torch.roll(y0, -1)
    inside = torch.zeros(points.shape[0], dtype=torch.bool)
    for i in range(polygon.shape[0]):
        dy = y1[i] - y0[i]
        if dy == 0:
            continue
        straddles = (y0[i] > y) != (y1[i] > y)
        x_at_y = x0[i] + (x1[i] - x0[i]) * (y - y0[i]) / dy
        inside ^= straddles & (x < x_at_y)
    return inside

def check_points_in_polygon():
    square = torch.tensor([[0.0, 0.0], [4.0, 0.0], [4.0, 4.0], [0.0, 4.0]])
    u = torch.tensor(
        [
            [0.0, 0.0],
            [5.0, 0.0],
            [5.0, 5.0],
            [3.0, 5.0],
            [3.0, 2.0],
            [2.0, 2.0],
            [2.0, 5.0],
            [0.0, 5.0],
        ]
    )
    pts = torch.tensor(
        [
            [2.0, 2.0],
            [5.0, 1.0],
            [-1.0, 0.0],
            [1.0, 4.0],
            [2.5, 4.0],
            [2.5, 1.0],
            [4.0, 4.0],
        ]
    )
    assert torch.equal(points_in_polygon(pts[:3], square), torch.tensor([True, False, False]))
    assert torch.equal(points_in_polygon(pts[3:], u), torch.tensor([True, False, True, True]))
    print("passed")

check_points_in_polygon()


## 4. Douglas–Peucker

Simplify an open polyline. Always keep the endpoints. Keep an intermediate vertex when its perpendicular distance to the current segment exceeds `epsilon`. A point at exactly `epsilon` is dropped. Preserve input order.

A loop or a stack is appropriate. The distance test is the vectorized part.

Hand check: `(0,0), (1, 0.05), (2, -0.05), (3, 0)` with `epsilon = 0.1` collapses to the two endpoints. The spike `(0,0), (1,0), (1,1), (2,0)` with `epsilon = 0.8` keeps `(0,0), (1,1), (2,0)`. The same spike with `epsilon = 0.1` keeps every vertex.

<details><summary>Hint</summary>

Put `(0, n-1)` on a stack. On each span, find the farthest vertex between the ends. If it is farther than `epsilon`, keep it and push both sub-spans.

</details>


In [ ]:
import torch

def douglas_peucker(points, epsilon):
    """
    Simplify an open polyline. Keep a vertex when its deviation exceeds epsilon.

    points: (M, D). A point exactly epsilon away is dropped (`>` rather than `>=`).
    Endpoints are always kept. Output order follows the input.
    """
    raise NotImplementedError


### Solution

Reference answer and checks. It replaces whatever you defined above.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_default_dtype(torch.float64)
DTYPE = torch.float64

def _t(x, dtype=DTYPE):
    if isinstance(x, torch.Tensor):
        return x.to(dtype=dtype)
    return torch.as_tensor(x, dtype=dtype)

def douglas_peucker(points, epsilon):
    """Simplify an open polyline. Drop a vertex at exactly epsilon."""
    points = _t(points)

    def distances_to_segment(pts, a, b):
        ab = b - a
        denom = torch.dot(ab, ab)
        if denom < 1e-24:
            return torch.linalg.norm(pts - a, dim=1)
        t = torch.clamp(((pts - a) @ ab) / denom, 0.0, 1.0)
        closest = a + t[:, None] * ab
        return torch.linalg.norm(pts - closest, dim=1)

    n = points.shape[0]
    if n <= 2:
        return points.clone()
    keep = torch.zeros(n, dtype=torch.bool)
    keep[0] = True
    keep[-1] = True
    stack = [(0, n - 1)]
    while stack:
        i, j = stack.pop()
        if j <= i + 1:
            continue
        dist = distances_to_segment(points[i + 1 : j], points[i], points[j])
        k = int(torch.argmax(dist))
        if dist[k] > epsilon:
            idx = i + 1 + k
            keep[idx] = True
            stack.append((i, idx))
            stack.append((idx, j))
    return points[keep]

def check_douglas_peucker():
    line = torch.tensor([[0.0, 0.0], [1.0, 0.05], [2.0, -0.05], [3.0, 0.0]])
    assert torch.allclose(douglas_peucker(line, 0.1), torch.tensor([[0.0, 0.0], [3.0, 0.0]]))
    spike = torch.tensor([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [2.0, 0.0]])
    assert torch.allclose(douglas_peucker(spike, 0.8), torch.tensor([[0.0, 0.0], [1.0, 1.0], [2.0, 0.0]]))
    assert torch.allclose(douglas_peucker(spike, 0.1), spike)
    print("passed")

check_douglas_peucker()
